In [ ]:
!pip install opencv-python

### Image processing

In [5]:
import cv2
import time
import numpy as np

cam = cv2.VideoCapture(0)

start_time = time.time()

bpm = 120

while (True):
    success, frame = cam.read()
    
    if (not success):
        break
    
    curr_time = time.time()
    
    num = int((curr_time - start_time) / (60.0 / bpm))
    
    if (num % 4 == 0):
        frame = 255 - frame

    if (num % 4 == 1):
        frame[:, :, 1] += 50
    
    if (num % 4 == 2):
        frame[:, :, 0] = 250

    if (num % 4 == 3):
        shift = 1 + np.random.randint(35)
        frame[:, 0 : -shift, 2] = frame[:, shift : , 2]
    
    cv2.imshow("image", frame)
    #cv2.imshow("image_b", frame[:, :, 0])
    #cv2.imshow("image_g", frame[:, :, 1])
    #cv2.imshow("image_r", frame[:, :, 2])
    #cv2.imshow("crop", frame[200 : 500, 0 : 500, :])
    
    #print(frame.shape, frame[100, 200, 1])
    
    key = cv2.waitKey(100) & 0xFF
    
    if (key == ord('q')):
        break

cam.release()
cv2.destroyAllWindows()
cv2.waitKey(10)

-1

### Color-based detection

In [ ]:
import cv2
import numpy as np

cv2.namedWindow("mask")

#def print_new_value(new_value):
#    print(new_value)

#cv2.createTrackbar("lb", "mask", 0, 255, print_new_value)
cv2.createTrackbar("lb", "mask", 72, 255, lambda i : None)
cv2.createTrackbar("hb", "mask", 168, 255, lambda i : None)
cv2.createTrackbar("lg", "mask", 14, 255, lambda i : None)
cv2.createTrackbar("hg", "mask", 35, 255, lambda i : None)
cv2.createTrackbar("lr", "mask", 0, 255, lambda i : None)
cv2.createTrackbar("hr", "mask", 0, 255, lambda i : None)

camera = cv2.VideoCapture(0)

while(True):
    success, frame_ = camera.read()
    
    frame = cv2.resize(frame_, (960, 540))
    
    #hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    #             input image (low blue, low green, ...) (high blue, ...)
    
    lb = cv2.getTrackbarPos("lb", "mask")
    hb = cv2.getTrackbarPos("hb", "mask")
    lg = cv2.getTrackbarPos("lg", "mask")
    hg = cv2.getTrackbarPos("hg", "mask")
    lr = cv2.getTrackbarPos("lr", "mask")
    hr = cv2.getTrackbarPos("hr", "mask")
    
    #print(f"low blue: {lblue}")
    
    mask = cv2.inRange(frame, (lb, lg, lr), (hb, hg, hr))
    
    ker = np.ones((13, 13), np.uint8)
    dilated = cv2.dilate(mask, ker)
    
    output = cv2.connectedComponentsWithStats(dilated)
    
    num_components = output[0]
    stats = output[2]
    
    for i in range(num_components):
        top = stats[i, cv2.CC_STAT_TOP]
        left = stats[i, cv2.CC_STAT_LEFT]
        width = stats[i, cv2.CC_STAT_WIDTH]
        height = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]
        
        cv2.rectangle(frame, (left, top), (left + width, top + height), (123, 234, 234), 3)

        image = cv2.putText(frame, str(area), (left, top), cv2.FONT_HERSHEY_SIMPLEX, 
                        1, (255, 0, 0), 2, cv2.LINE_AA)
    
    cv2.imshow("image", frame)
    cv2.imshow("mask", mask)
    cv2.imshow("dilated", dilated)
    
    #cv2.imshow("hsv", hsv)
    #cv2.imshow("hsv0", hsv[:, :, 0])
    #cv2.imshow("hsv1", hsv[:, :, 1])
    #cv2.imshow("r2", frame[:, :, 2])
    
    key = cv2.waitKey(100)
    
    if (key == ord('q')):
        break

camera.release()
cv2.destroyAllWindows()
cv2.waitKey(100)

### HSV distribution manipulation

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def plot_dist(channel):
    fig, ax = plt.subplots()
    ax.hist(channel.ravel(), 25, [0,256])
    
    fig.canvas.draw()
    dist = np.array(fig.canvas.renderer.buffer_rgba())
    plt.close()
    
    return dist

cv2.namedWindow("mask")

cv2.createTrackbar("h", "mask", 256, 511, lambda i : None)
cv2.createTrackbar("s", "mask", 256, 511, lambda i : None)
cv2.createTrackbar("v", "mask", 256, 511, lambda i : None)

camera = cv2.VideoCapture(0)

while(True):
    success, frame_ = camera.read()
    
    frame = cv2.resize(frame_, (960, 540))
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
    h = cv2.getTrackbarPos("h", "mask")
    s = cv2.getTrackbarPos("s", "mask")
    v = cv2.getTrackbarPos("v", "mask")
    
    #hsv[:, :, 0] += h - 256
    #hsv[:, :, 1] += s - 256
    #hsv[:, :, 2] += v - 256
    
    hsv[:, :, 0] = cv2.addWeighted(hsv[:, :, 0], 1, h - 256, 1, 0)
    hsv[:, :, 1] = cv2.addWeighted(hsv[:, :, 1], 1, s - 256, 1, 0)
    hsv[:, :, 2] = cv2.addWeighted(hsv[:, :, 2], 1, v - 256, 1, 0)
    
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    
    val_hist = plot_dist(hsv[::5, ::5, 2])
    
    cv2.imshow("image", frame)
    cv2.imshow("hsv", hsv)
    cv2.imshow("modified", bgr)
    cv2.imshow("val hist", val_hist)
    
    key = cv2.waitKey(100)
    
    if (key == ord('q')):
        break

camera.release()
cv2.destroyAllWindows()
cv2.waitKey(100)

### Morphological operations

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

arr = np.zeros((12, 12), np.uint8)

arr[2:10, 3:7] = 255
arr[7:8, 1:9] = 255
arr[2:3, 9:11] = 255

ker = np.ones((3, 3), np.uint8)

eroded = cv2.erode(arr, ker)
dilated = cv2.dilate(eroded, ker)

#ker[1, 2] = 0
print(ker)

#plt.imshow(ker, cmap="gray")
#plt.show()
plt.imshow(arr, cmap="gray")
plt.show()
plt.imshow(eroded, cmap="gray")
plt.show()
plt.imshow(dilated, cmap="gray")
plt.show()

### Background subtraction

In [ ]:
import cv2
import numpy as np

cv2.namedWindow("mask")

#def print_new_value(new_value):
#    print(new_value)

#cv2.createTrackbar("lb", "mask", 0, 255, print_new_value)
cv2.createTrackbar("lb", "mask", 0, 255, lambda i : None)
cv2.createTrackbar("hb", "mask", 255, 255, lambda i : None)
cv2.createTrackbar("lg", "mask", 0, 255, lambda i : None)
cv2.createTrackbar("hg", "mask", 255, 255, lambda i : None)
cv2.createTrackbar("lr", "mask", 30, 255, lambda i : None)
cv2.createTrackbar("hr", "mask", 255, 255, lambda i : None)

filename = "unicycle.mp4"
camera = cv2.VideoCapture(filename)

IMGX, IMGY = 960, 540

background = np.zeros((IMGY, IMGX, 3), np.uint8)
alpha = 0.98

while(True):
    success, frame_ = camera.read()
    
    if (not success):
        camera.release()
        camera = cv2.VideoCapture(filename)
        continue
    
    frame = cv2.resize(frame_, (IMGX, IMGY))
    
    #backgr = backgr * alpha + new_frame * (1 - alpha) + 0
    
    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    background = cv2.addWeighted(background, alpha, hsv_frame, 1 - alpha, 0)
    
    diff = cv2.absdiff(background, hsv_frame)
    
    lb = cv2.getTrackbarPos("lb", "mask")
    hb = cv2.getTrackbarPos("hb", "mask")
    lg = cv2.getTrackbarPos("lg", "mask")
    hg = cv2.getTrackbarPos("hg", "mask")
    lr = cv2.getTrackbarPos("lr", "mask")
    hr = cv2.getTrackbarPos("hr", "mask")
    
    mask = cv2.inRange(diff, (lb, lg, lr), (hb, hg, hr))
    
    ker = np.ones((3, 3), np.uint8)
    eroded = cv2.erode(mask, ker)

    ker = np.ones((65, 35), np.uint8)
    dilated = cv2.dilate(eroded, ker)
    
    output = cv2.connectedComponentsWithStats(dilated)
    
    num_components = output[0]
    stats = output[2]
    
    for i in range(1, num_components):
        top = stats[i, cv2.CC_STAT_TOP]
        left = stats[i, cv2.CC_STAT_LEFT]
        width = stats[i, cv2.CC_STAT_WIDTH]
        height = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]

        if (area > 4000):
            cv2.rectangle(frame, (left, top), (left + width, top + height), (123, 234, 234), 3)

            image = cv2.putText(frame, str(area), (left, top), cv2.FONT_HERSHEY_SIMPLEX, 
                            1, (255, 0, 0), 2, cv2.LINE_AA)
    
    cv2.imshow("frame", frame)
    #cv2.imshow("image", hsv_frame)
    #cv2.imshow("background", background)

    #cv2.imshow("diff", diff)
    #cv2.imshow("diff_hue", diff[:, :, 0])
    #cv2.imshow("diff_saturation", diff[:, :, 1])
    #cv2.imshow("diff_value", diff[:, :, 2])

    cv2.imshow("mask", mask)
    cv2.imshow("dilated", dilated)
    cv2.imshow("eroded", eroded)
        
    key = cv2.waitKey(100)
    
    if (key == ord('q')):
        break

camera.release()
cv2.destroyAllWindows()
cv2.waitKey(100)

### Contours and adaptive thresholding

In [ ]:
import matplotlib.pyplot as plt
import cv2

img = cv2.imread("wall_img.jpg")

print(img.shape[0] // 2, img.shape[1] // 2)

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

plt.imshow(img)
plt.show()
plt.imshow(hsv)
plt.show()
plt.imshow(hsv[:, :, 0])
plt.show()
plt.imshow(hsv[:, :, 1])
plt.show()
plt.imshow(hsv[:, :, 2])
plt.show()


In [ ]:
import cv2
import numpy as np

cv2.namedWindow("mask")

cv2.createTrackbar("lh", "mask", 0, 255, lambda i : None)
cv2.createTrackbar("hh", "mask", 255, 255, lambda i : None)
cv2.createTrackbar("ls", "mask", 0, 255, lambda i : None)
cv2.createTrackbar("hs", "mask", 255, 255, lambda i : None)
cv2.createTrackbar("lv", "mask", 0, 255, lambda i : None)
cv2.createTrackbar("hv", "mask", 255, 255, lambda i : None)

img = cv2.imread("wall_img.jpg")

while(True):    
    frame = cv2.resize(img, (1280, 961))
    
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    lh = cv2.getTrackbarPos("lh", "mask")
    hh = cv2.getTrackbarPos("hh", "mask")
    ls = cv2.getTrackbarPos("ls", "mask")
    hs = cv2.getTrackbarPos("hs", "mask")
    lv = cv2.getTrackbarPos("lv", "mask")
    hv = cv2.getTrackbarPos("hv", "mask")
        
    mask = cv2.inRange(frame, (lh, ls, lv), (hh, hs, hv))
    
    # ker = np.ones((13, 13), np.uint8)
    # dilated = cv2.dilate(mask, ker)
    
    # output = cv2.connectedComponentsWithStats(dilated)
    
    # num_components = output[0]
    # stats = output[2]
    
    # for i in range(num_components):
    #     top = stats[i, cv2.CC_STAT_TOP]
    #     left = stats[i, cv2.CC_STAT_LEFT]
    #     width = stats[i, cv2.CC_STAT_WIDTH]
    #     height = stats[i, cv2.CC_STAT_HEIGHT]
    #     area = stats[i, cv2.CC_STAT_AREA]
        
    #     cv2.rectangle(frame, (left, top), (left + width, top + height), (123, 234, 234), 3)

    #     image = cv2.putText(frame, str(area), (left, top), cv2.FONT_HERSHEY_SIMPLEX, 
    #                     1, (255, 0, 0), 2, cv2.LINE_AA)
    
    cv2.imshow("image", frame)
    cv2.imshow("mask", mask)
    #cv2.imshow("dilated", dilated)
    
    #cv2.imshow("hsv", hsv)
    #cv2.imshow("hsv0", hsv[:, :, 0])
    #cv2.imshow("hsv1", hsv[:, :, 1])
    #cv2.imshow("r2", frame[:, :, 2])
    
    key = cv2.waitKey(100)
    
    if (key == ord('q')):
        break

cv2.destroyAllWindows()
cv2.waitKey(100)

In [ ]:
value = hsv[:, :, 2]

mask = cv2.adaptiveThreshold(value, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 35, 0)

# plt.imshow(value)
# plt.show()
plt.imshow(mask)
plt.show()

In [ ]:
import cv2
import numpy as np

cv2.namedWindow("mask")

cv2.createTrackbar("bs", "mask", 19, 205, lambda i : None)
cv2.createTrackbar("th", "mask", 40, 255, lambda i : None)
cv2.createTrackbar("bin type", "mask", 0, 1, lambda i : None)

img = cv2.imread("wall_img.jpg")

while(True):    
    frame = cv2.resize(img, (1280, 961))
    
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    block_size = cv2.getTrackbarPos("bs", "mask") * 2 + 3
    th = cv2.getTrackbarPos("th", "mask") - 20
    bin_type = cv2.getTrackbarPos("bin type", "mask")

    if (bin_type == 0):
        mask = cv2.adaptiveThreshold(hsv[:, :, 2], 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, block_size, th)
    
    else:
        mask = cv2.adaptiveThreshold(hsv[:, :, 2], 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, block_size, th)
    
    cv2.imshow("image", frame)
    cv2.imshow("mask", mask)
    
    cv2.imshow("hsv2", hsv[:, :, 2])
    
    key = cv2.waitKey(100)
    
    if (key == ord('q')):
        break

cv2.destroyAllWindows()
cv2.waitKey(100)

In [ ]:
def plot_dist(channel):
    fig, ax = plt.subplots()
    ax.hist(channel.ravel(), 255, [0,256])
    
    fig.canvas.draw()
    dist = np.array(fig.canvas.renderer.buffer_rgba())
    plt.close()
    
    return dist

hehehe = cv2.imread("bim.png")

#img = cv2.imread("wall_img.jpg")

hsv = cv2.cvtColor(hehehe, cv2.COLOR_BGR2HSV)
value = hsv[:, :, 2]
plt.imshow(value)
plt.show()

hist = plot_dist(value)
plt.imshow(hist)
plt.show()

blur = cv2.GaussianBlur(value, (13, 13), 0)

th, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print(f"th = {th}")

#cv2.medianBlur(value, 7)
hist = plot_dist(blur)

plt.imshow(hist)
plt.show()
plt.imshow(mask)
plt.show()

In [ ]:
contours, hierarchy = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(len(contours))

img = cv2.imread("bim.png")

for cnt in contours:
    length = cv2.arcLength(cnt, True)
    area   = cv2.contourArea(cnt)
    
    if (length > 200):
        cv2.drawContours(img, [cnt], 0, (0, 255, 0), 3)
        
        x_avg = int(np.mean(cnt[:, :, 0]))
        y_avg = int(np.mean(cnt[:, :, 1]))
        
        #print(x_avg, y_avg)
        
        # image = cv2.putText(img, str(int(area)), (x_avg, y_avg), cv2.FONT_HERSHEY_SIMPLEX, 
        #                     1, (255, 0, 255), 2, cv2.LINE_AA)

        # image = cv2.putText(img, str(int(length)), (x_avg, y_avg + 40), cv2.FONT_HERSHEY_SIMPLEX, 
        #                     1, (255, 0, 255), 2, cv2.LINE_AA)

        image = cv2.putText(img, str(int(area / length)), (x_avg, y_avg), cv2.FONT_HERSHEY_SIMPLEX, 
                            1, (255, 0, 255), 2, cv2.LINE_AA)
        
        epsilon = 0.04 * cv2.arcLength(cnt, True)
        
        approx = cv2.approxPolyDP(cnt, epsilon, True)

        cv2.drawContours(img, [approx], 0, (255, 255, 0), 3)

plt.imshow(img)
plt.show()

In [ ]:
cnt = contours[0]

print(cnt)

x_avg = np.mean(cnt[:, :, 0])

print(x_avg)